# Finance and Statistics / Monte Carlo Simulation Engine / Stochastic Modeling via GBM

#### Abstract
Este repositorio implementa un motor de Simulación de Monte Carlo diseñado para la estimación de trayectorias potenciales en activos de renta variable. Partiendo de la premisa epistemológica de que la predicción determinista es una falacia en sistemas complejos, este proyecto desplaza el enfoque hacia la cuantificación de la incertidumbre. El objetivo no es señalar un precio futuro único, sino modelar una distribución de densidades que revele el espectro de resultados probables basados en la volatilidad histórica y la deriva del activo.

**Palabras Clave: _simulación montecarlo, movimiento browniano geométrico, estimar vs predecir_**

### INTRODUCCIÓN
Sostengo la convicción de que "predecir" es un término inapropiado; en su defecto el concepto más preciso es estimar. La literatura sobre sistemas y modelos acuerda que lo posible es estimar el comportamiento agregado de un conjunto de variables o eventos aleatorios independientes mas no predecir de manera determinista.

Herramientas como las simulaciones de Monte Carlo permiten estimar un valor razonable de un activo mediante la simulación de miles de trayectorias posibles, extrayendo el promedio como métrica de referencia. Por otro lado, los modelos ocultos de Markov se utilizan para inferir estados. Observando patrones en los datos observados. De manera que el proposito verdadero de modelar es proporcionar un marco informado para la toma decisiones bajo incertidumbre cuantificando una ventaja estadistica.

### MARCO TEÓRICO
Bajo esta óptica, las generalizaciones dejan de ser etiquetas para transformarse en estimaciones de comportamiento agregado. No pretendemos describir la acción individual final, sino estimar la respuesta del sistema basándonos en sus patrones recurrentes.

Esta distinción es crucial para no caer en la trampa del relativismo. A menudo se confunde la ausencia de una verdad absoluta con la falta de una estructura objetiva; sin embargo, técnicamente no nos enfrentamos a una "verdad relativa", sino a una incerteza inherente. La realidad objetiva no se manifiesta en el evento único que el humano intenta "predecir", sino en la consistencia de las leyes que permiten su estimación. En este contexto, la subjetividad no es un punto de vista válido, sino simplemente el ruido en la medición de un fenómeno que opera independientemente del observador.

La prueba irrefutable de esta mecánica subyacente reside en la estabilidad de las distribuciones. La evidencia de lo real no se encuentra en la capacidad de adivinar el futuro, sino en la precisión con la que podemos calcular el margen de error y la dispersión. Es la consistencia del azar, este motor que genera grandes volúmenes de resultados probables, lo que confirma que existe una estructura sólida debajo del caos aparente.

### MARCO METODOLÓGICO
Debemos entender los fenómenos como mecánica sin narrativa. Los eventos no son una cuestión de apreciación artística o interpretación moral, sino transiciones de estado en un sistema físico o biológico. Mientras que la "predicción" es una narrativa humana —un intento de imponer orden y sentido mediante el lenguaje—, la estimación estocástica trata al fenómeno por lo que es: una causalidad mecánica. Nuestra capacidad de análisis se limita, por tanto, a cartografiar la probabilidad de una ocurrencia dentro de un espectro de posibilidades, despojando al evento de su historia y devolviéndole su naturaleza de proceso.

Para dicha tarea es necesario armarse de un algoritmo con 4 responsabilidades:
1. Extracción y saneamiento de datos historicos (OHLCV) usando una libreria externa (yfinance, interactive brokers)
2. Cálculo de la media logarítmica ($\mu$) y la desviación estándar ($\sigma$) de los retornos diarios para definir el perfil de riesgo-retorno del activo.
3. Ejecución de $n$ simulaciones (ej. 10,000 iteraciones) sobre un horizonte temporal definido. El motor utiliza el Movimiento Browniano Geométrico (GBM), asumiendo que los cambios en el precio siguen una distribución log-normal, lo que permite capturar la naturaleza estocástica del mercado.
4. Procesamiento de los resultados finales.

La implementación se apoya en el ecosistema científico de Python: pandas para el gobierno de datos, numpy para el cómputo vectorial, y scipy.stats para la modelización de funciones de densidad normal. La capa visual se gestiona mediante matplotlib, permitiendo una interpretación intuitiva de la varianza simulada.

### RESULTADOS
El proyecto está diseñado para ser modular. El usuario puede orquestar la simulación modificando el ticker del activo, ajustando la densidad de las simulaciones (num_simulations) o expandiendo el horizonte temporal (num_days) para observar la degradación de la certidumbre a largo plazo. Para el analisis de resultados, los precios finales de todas las simulaciones son computados para hallar la mediana, los intervalos de confianza e identificar la trayectoria de precio más probable. Las trayectorias de precio son graficadas para representar el rango de futuros precios potenciales.

### REFERENCIAS: {
    https://youtu.be/fO-lGzZADVU , 14 minute video that inspired this python project.
    https://youtu.be/-4sf43SLL3A , for proof reading and revisioning.
}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
import logging
from typing import List, Dict, Tuple, Optional

def setup_logging(level=logging.INFO):
    
    Configures the logging for the notebook.
    This helps in controlling output and avoiding sensitive information in stdout/stderr.
    logging.basicConfig(level=level, format='%(asctime)s - %(levelname)s - %(message)s')
    # Suppress yfinance messages if desired, though 'progress=False' already helps
    logging.getLogger('yfinance').setLevel(logging.CRITICAL)

setup_logging()
logger = logging.getLogger(__name__)
sns.set_theme(style="darkgrid")

In [ ]:
# --- GLOBAL CONFIGURATION AND CONSTANTS ---\
START_DATE = '1950-01-01'
NUM_SIMULATIONS = 10000
NUM_DAYS_SINGLE_TICKER = 252  # Prediction horizon for single ticker (e.g., 1 year of trading days)\
NUM_DAYS_MULTIPLE_TICKERS = 22 # Prediction horizon for multiple tickers (e.g., 1 month of trading days)\
# --- DATA ACQUISITION FUNCTIONS ---\
def get_stock_data(\
    ticker_symbol: str | List[str], \
    start_date: str = START_DATE
) -> Optional[pd.DataFrame]:
    
    Downloads historical stock data for a given ticker or list of tickers.

    Args:
        ticker_symbol (str | List[str]): The stock ticker symbol(s) (e.g., 'PLTR', ['MSFT', 'GOOGL']).
        start_date (str): The start date for historical data (format 'YYYY-MM-DD').

    Returns:
        Optional[pd.DataFrame]: A DataFrame with the historical data, or None if an error occurs.
    
    try:
        data = yf.download(\
            ticker_symbol,\
            start=start_date,\
            group_by='ticker' if isinstance(ticker_symbol, list) else 'default',
            progress=False,\
            # actions=True # To include dividends and stock splits if needed
        )\
        if data.empty:\
            logger.warning(f"No data retrieved for ticker(s): {ticker_symbol}")
            return None
        logger.info(f"Successfully downloaded data for {ticker_symbol} from {start_date}.")
        return data
    except Exception as e:\
        logger.error(f"Error downloading data for {ticker_symbol}: {e}")
        return None

def calculate_returns(df: pd.DataFrame, ticker: str | Tuple[str, ...]) -> pd.DataFrame:
    
    Calculates the percentage change (returns) for the 'Close' price of a stock.
    Handles both single and multi-level columns from yfinance.

    Args:
        df (pd.DataFrame): DataFrame containing stock data.
        ticker (str | Tuple[str, ...]): The ticker symbol or tuple representing the column.

    Returns:
        pd.DataFrame: DataFrame with a new 'Returns' column.
    
    if isinstance(ticker, str): # Single ticker case
        close_col = ('Close',)
        if close_col not in df.columns: # Adjust for yfinance default output for single ticker (no multi-index)
            close_col = 'Close'
    elif isinstance(ticker, tuple): # Multi-ticker case, yfinance creates multi-index columns
        close_col = (ticker[0], 'Close') # Assuming ticker is ('PLTR',) -> ('PLTR', 'Close')
    else: # Fallback for mis_tickers iteration which passes string
        close_col = (ticker, 'Close')
    
    # Check if the column exists before calculating returns
    if close_col in df.columns:
        # Address FutureWarning by explicitly setting fill_method=None or handling NaNs beforehand
        df['Returns'] = df[close_col].pct_change(fill_method=None)\
        logger.info(f"Calculated returns for {ticker}.")
    else:
        logger.error(f"Close price column {close_col} not found for {ticker}.")
    return df

# Un Ticker

In [ ]:
ticker = "PLTR"
df = yf.download(ticker, start='1950-01-01', group_by='ticker', progress=False)
# Calcular los retornos
df["Returns"] = df[('PLTR', 'Close')].pct_change()


In [ ]:
num_simulations=10000
num_days=252 #predict horizon
last_price=df[('PLTR', 'Close')].iloc[-1]

simulation_df=np.zeros((num_days,num_simulations))
mu=df["Returns"].mean()
sigma=df["Returns"].std()

In [ ]:
for i in range(num_simulations):
  price_list=[last_price]
  for j in range(num_days):
    price=price_list[-1] * np.exp((mu - 0.5 * sigma**2) + sigma * np.random.normal())
    price_list.append(price)
  simulation_df[:,i]=price_list[1:]

final_prices=simulation_df[-1,:]
median_final_prices=np.median(final_prices)

most_likely_price_index=np.argmin(np.abs(final_prices-median_final_prices))
most_likely_price_simulation=simulation_df[:,most_likely_price_index]

In [ ]:
plt.figure(figsize=(10,7))
plt.plot(simulation_df, alpha=0.25)
plt.title("Monte Carlo Simulation")
plt.xlabel("Days")
plt.ylabel("Price")
plt.show()

print(f"Resumen de la simulación para {ticker}:")
print(f"- Precio más probable al final del horizonte: {most_likely_price_simulation[-1]:.2f}")
print(f"- Mediana de precios finales: {median_final_prices:.2f}")


# Varios Tickers

In [ ]:
mis_tickers = ["PLTR",
               "MSFT",
               "GOOGL",
               "META",
               "LMT",
               "CVX"]

df2 = yf.download(mis_tickers, start='1950-01-01', group_by='ticker', progress=False)
for ticker in mis_tickers:
    df2[(ticker, 'Returns')] = df2[(ticker, 'Close')].pct_change()

In [ ]:
num_simulations = 10000
num_days = 22  # predict horizon

# Dictionaries to store results for each ticker
all_ticker_simulation_dfs = {}
all_ticker_most_likely_prices = {}
all_ticker_median_final_prices = {}

In [ ]:
print(f"--------- Simulating for {mis_tickers} ---------")
for current_ticker in mis_tickers:
        # Get ticker-specific parameters
    last_price = df2[(current_ticker, 'Close')].iloc[-1]

    # Calculate mu and sigma for the current ticker
    ticker_returns = df2[(current_ticker, 'Returns')].dropna() # Drop NaN for calculation
    mu = ticker_returns.mean()
    sigma = ticker_returns.std()

    # Initialize simulation_df for the current ticker
    simulation_df = np.zeros((num_days, num_simulations))

    # Run Monte Carlo simulation
    for i in range(num_simulations):
        price_list = [last_price]
        for j in range(num_days):
            price = price_list[-1] * np.exp((mu - 0.5 * sigma**2) + sigma * np.random.normal())
            price_list.append(price)
        simulation_df[:, i] = price_list[1:]

    # Analyze results
    final_prices = simulation_df[-1, :]
    median_final_prices = np.median(final_prices)
    most_likely_price_index = np.argmin(np.abs(final_prices - median_final_prices))
    most_likely_price_simulation = simulation_df[:, most_likely_price_index]

    # Store results
    all_ticker_simulation_dfs[current_ticker] = simulation_df
    all_ticker_most_likely_prices[current_ticker] = most_likely_price_simulation[-1]
    all_ticker_median_final_prices[current_ticker] = median_final_prices

    # Plotting for the current ticker
    plt.figure(figsize=(10, 7))
    plt.plot(simulation_df, alpha=0.2)
    plt.plot(most_likely_price_simulation, color='red', linestyle='--', label=f'Most Likely Path ({current_ticker})')
    plt.title(f"Monte Carlo Simulation for {current_ticker}")
    plt.xlabel("Days")
    plt.ylabel("Price")
    plt.legend()
    plt.show()

    print(f"-------- Results for {current_ticker} ---------")
    print(f"Median Final Simulated Price: ${all_ticker_median_final_prices[current_ticker]:.2f}")
    print(f"Most Likely Final Simulated Price: ${all_ticker_most_likely_prices[current_ticker]:.2f}")
    print("-" * 75)


In [ ]:
# Consolidated summary of results for all tickers
print(f"--- Consolidated Analysis for All Tickers (Prediction Horizon: {num_days} days, Simulations: {num_simulations}) ---")
for ticker in mis_tickers:
    simulated_price = all_ticker_most_likely_prices[ticker]
    print(f"{ticker}: Most Likely Simulated Final Price: {simulated_price:.3f}")

In [ ]:

# Create a list to store the results
results = []

# Collect data for each ticker
for ticker in mis_tickers:
    simulated_price = all_ticker_most_likely_prices[ticker]
    results.append({
        "Ticker": ticker,
        "Most Likely Simulated Final Price": round(simulated_price, 3)
    })

# Create a DataFrame from the results
results_df = pd.DataFrame(results)

# Display the table
print(f"--- Consolidated Analysis for All Tickers (Prediction Horizon: {num_days} days, Simulations: {num_simulations}) ---")
print(results_df.to_string(index=False))

## Distribución de Precios Finales Simulados

In [ ]:
for ticker in mis_tickers:
    plt.figure(figsize=(10, 6))
    final_prices = all_ticker_simulation_dfs[ticker][-1, :]
    sns.histplot(final_prices, bins=50, kde=True, color='skyblue')
    plt.title(f'Distribución de Precios Finales Simulados para {ticker}')
    plt.xlabel('Precio Final')
    plt.ylabel('Frecuencia')
    plt.axvline(all_ticker_median_final_prices[ticker], color='red', linestyle='--', label=f'Mediana: {all_ticker_median_final_prices[ticker]:.2f}')
    plt.legend()
    plt.grid(True, alpha=0.7)
    plt.show()

## Intervalo de Confianza del 95% para los Precios Finales Simulados

In [ ]:
print("--- 95% Confidence Interval for Simulated Final Prices ---")
for ticker in mis_tickers:
    final_prices = all_ticker_simulation_dfs[ticker][-1, :]
    lower_bound = np.percentile(final_prices, 2.5)
    upper_bound = np.percentile(final_prices, 97.5)
    print(f"{ticker}: 95% CI: [{lower_bound:.3f}, {upper_bound:.3f}]")